# TP 3 — Partie 5 : surveiller et observer la plateforme

Deux questions différentes, deux familles d'outils, que l'on retrouve dans toutes les équipes data :

| | **Monitoring** (supervision) | **Observabilité des données** |
|---|---|---|
| La question | Les **services** tournent-ils bien ? | Les **données** sont-elles fiables, et d'où viennent-elles ? |
| Ce qu'on regarde | processeur, mémoire, durée et échecs des tâches | *lineage* (qui produit quoi, à partir de quoi), historique des exécutions, schémas |
| Ici | **Prometheus** (collecte des métriques) + **Grafana** (tableaux de bord, alertes) | **OpenLineage** (format standard des événements) + **Marquez** (stockage et interface) |

Tout est déjà configuré : dans cette partie, vous **utilisez** les outils, comme une équipe d'exploitation.

> **⚠️ Mémoire.** Arrêtez Trino et Superset, puis démarrez les outils de supervision depuis le terminal **Ubuntu** :
> ```bash
> docker compose --profile full stop trino superset
> docker compose --profile observabilite --profile orchestration up -d
> ```
> Relancez ensuite le DAG `catalogue_photos` dans Airflow (http://localhost:8085) pour produire des métriques et du lineage.

In [ ]:
import requests
import pandas as pd

PROMETHEUS, MARQUEZ = "http://prometheus:9090", "http://marquez:5000"


def promql(requete):
    """Exécute une requête PromQL et renvoie un DataFrame (une ligne par série)."""
    r = requests.get(f"{PROMETHEUS}/api/v1/query", params={"query": requete}, timeout=30).json()
    return pd.DataFrame([{**s["metric"], "valeur": float(s["value"][1])} for s in r["data"]["result"]])


def a_completer(quoi):
    """Signale clairement une partie du TP qui n'a pas encore été écrite."""
    raise NotImplementedError(f"⚠️ Cellule à compléter : {quoi}")

## Étape 1 — Prometheus : des métriques, interrogées en PromQL

Prometheus interroge chaque source de métriques toutes les 15 secondes et garde l'historique. Une **métrique** est une série de valeurs dans le temps, décrite par des **étiquettes** (`name` du conteneur, `dag_id`, `task_id`…). Le langage de requête s'appelle **PromQL**.

In [ ]:
memoire = promql('container_memory_working_set_bytes{name=~"lakehouse-.*"}')
memoire.assign(mo=(memoire.valeur / 1e6).round()).sort_values("mo", ascending=False)[["name", "mo"]]

Écrivez la requête PromQL qui donne la **durée médiane de chaque tâche** du DAG `catalogue_photos` : métrique `airflow_tache_duree_secondes`, étiquettes `dag_id="catalogue_photos"` et `quantile="0.5"`.

In [ ]:
durees = a_completer("requête PromQL")  # À COMPLÉTER : promql('...')
durees[["task_id", "valeur"]].sort_values("valeur", ascending=False)

## Étape 2 — Grafana : le tableau de bord et une alerte

Ouvrez http://localhost:3000 (`admin` / `admin`), menu *Dashboards > Lakehouse > Lakehouse — supervision*. Puis :
1. Relancez le DAG et observez, pendant l'exécution, la mémoire du conteneur `lakehouse-airflow-1`.
2. Créez une **alerte** (*Alerting > Alert rules > New alert rule*) : requête `sum(increase(airflow_taches_en_echec_total[5m]))`, condition « au-dessus de 0 », évaluée chaque minute.
3. Provoquez un échec, comme en partie 2 (seuil de rejet à 1 %), et vérifiez que l'alerte passe à l'état *Firing*.

> **❓ Question 1.** Quel conteneur consomme le plus de mémoire pendant le DAG ? Quelle tâche dure le plus longtemps ? Sur une machine de 8 Go, que pourriez-vous en conclure pour la configuration (`mem_limit`, `spark.driver.memory`) ?

## Étape 3 — Marquez : le lineage, construit automatiquement

Chaque job Spark lancé par Airflow envoie des événements **OpenLineage** à Marquez : ce qu'il a lu, ce qu'il a écrit, quand, avec quel résultat. Personne n'a dessiné le graphe : il est **déduit des exécutions réelles**.

Ouvrez http://localhost:3001, choisissez l'espace de noms `lakehouse`, puis cliquez sur le jeu de données `gold.photos_controle` : le graphe montre toute la chaîne qui le produit.

In [ ]:
jobs = requests.get(f"{MARQUEZ}/api/v1/namespaces/lakehouse/jobs", timeout=30).json()["jobs"]
pd.DataFrame([{"job": j["name"], "entrees": [i["name"] for i in j.get("inputs", [])],
               "sorties": [o["name"] for o in j.get("outputs", [])],
               "dernier_etat": (j.get("latestRun") or {}).get("state")} for j in jobs])

> **❓ Question 2.** Vous découvrez que `gold.photos_controle` contient des erreurs. Comment le lineage vous aide-t-il à trouver l'origine du problème ? Et, dans l'autre sens, à savoir qui sera touché si vous modifiez `silver.photos` ?

> **❓ Question 3.** Pour chaque situation, dites si c'est plutôt le monitoring ou l'observabilité des données qui vous alerte : (a) le conteneur Airflow manque de mémoire ; (b) le DAG réussit, mais un vendeur envoie depuis hier des photos en noir et blanc ; (c) `dbt_build` dure deux fois plus longtemps que d'habitude ; (d) une colonne a disparu de `bronze.produits_brut`.